In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0', 'google-generativeai>=0.8.0',
], check=True)

In [ ]:
import os, json, time, threading, uuid
from pathlib import Path
from datetime import datetime
import yaml, requests
import google.generativeai as genai
import google.api_core.exceptions
import sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')
from shared.gemini_rate_limiter import GeminiRateLimiter
from shared.secrets import load_secrets
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
SEEDS_PATH      = WORK_DIR / 'seed_dialogues.jsonl'
AUGMENTED_PATH  = WORK_DIR / 'augmented_dialogues.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p3b.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

VARIANTS_PER_SEED = 9
BATCH_SIZE        = 3
MAX_RETRIES       = 6
SAVE_EVERY        = 20
COSINE_SIM_THRESH = 0.92

In [ ]:
SECRETS = load_secrets(require_gemini=True)
HF_TOKEN         = SECRETS['HF_TOKEN_PRIMARY']
GEMINI_KEY = SECRETS.get('GEMINI_API_KEY_01') or SECRETS.get('GEMINI_API_KEY_02') or ''
genai.configure(api_key=GEMINI_KEY)
GEMINI_MODEL = genai.GenerativeModel("gemini-2.5-flash")
RATE_LIMITER = GeminiRateLimiter(rpm_limit=14)

with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE2_REPO = repos_cfg['repos']['stage2_moe']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage2: {STAGE2_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — augmented={state["stats"]["augmented"]}')
            return state
        except Exception: pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE2_REPO}/resolve/main/checkpoint_p3b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f: json.dump(state, f)
            print(f'[checkpoint] HF fallback — augmented={state["stats"]["augmented"]}')
            return state
    except Exception: pass
    print('[checkpoint] fresh start')
    return {'done_ids': [], 'stats': {'augmented': 0, 'deduped': 0, 'failed': 0}, 'last_updated': None}

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p3b.json', repo_id=STAGE2_REPO,
                repo_type='dataset', commit_message='p3b checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state    = load_checkpoint()
done_set = set(state['done_ids'])

In [ ]:
AUGMENT_SYSTEM = """You are a dialogue augmentation system for Pakistani Urdu customer support training.
Given a seed dialogue, generate paraphrase variants that preserve the intent and tool calls
but vary the wording, formality, and Urdu-English code-switching density.
Return ONLY a valid JSON array of dialogue objects with the same schema as the input.
Do NOT change the domain, difficulty, intents_covered, or tools_referenced fields.
Vary: user persona tone (polite/frustrated/urgent/confused), formality level, how much English is mixed in."""


def augment_seed(seed_dialogue, n_variants):
    user_msg = f'Generate exactly {n_variants} paraphrase variants of this dialogue:\n{json.dumps(seed_dialogue, ensure_ascii=False)}'
    for attempt in range(MAX_RETRIES):
        try:
            RATE_LIMITER.acquire()
            response = GEMINI_MODEL.generate_content(
                [AUGMENT_SYSTEM + "\n\n" + user_msg],
                generation_config={"response_mime_type": "application/json"}
            )
            raw = response.text.strip().replace('```json','').replace('```','').strip()
            variants = json.loads(raw)
            if not isinstance(variants, list): raise ValueError('Expected list')
            return variants
        except json.JSONDecodeError:
            if attempt == MAX_RETRIES - 1: return []
            time.sleep(5 * (attempt + 1))
        except Exception as e:
            if attempt == MAX_RETRIES - 1: print(f'  [augment] failed: {e}'); return []
            time.sleep(min(2**attempt, 60))
    return []


def simple_tfidf_dedup(texts, threshold=COSINE_SIM_THRESH):
    from collections import Counter
    import math
    def tokenize(t):
        return t.lower().split()
    def cosine(a, b):
        ca, cb = Counter(a), Counter(b)
        common = sum((ca & cb).values())
        if not common: return 0.0
        return common / math.sqrt(sum(ca.values()) * sum(cb.values()))
    kept, kept_tokens = [], []
    for text in texts:
        toks = tokenize(text)
        if all(cosine(toks, kt) < threshold for kt in kept_tokens):
            kept.append(text)
            kept_tokens.append(toks)
    return kept


In [ ]:
seeds = []
with open(SEEDS_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: seeds.append(json.loads(line))

pending = [s for s in seeds if s['id'] not in done_set]
print(f'[p3b] total seeds={len(seeds)} already done={len(done_set)} pending={len(pending)}')

with open(AUGMENTED_PATH, 'a', encoding='utf-8') as out_f:
    for idx, seed in enumerate(pending):
        seed_id  = seed['id']
        variants = augment_seed(seed, VARIANTS_PER_SEED)

        all_texts = [' '.join(t.get('text','') for t in v.get('turns',[])) for v in variants]
        kept_texts = simple_tfidf_dedup(all_texts)

        written = 0
        for v, text in zip(variants, all_texts):
            if text not in kept_texts:
                with cp_lock: state['stats']['deduped'] += 1
                continue
            v['id']           = f"{seed['domain']}_aug_{str(uuid.uuid4())[:8]}"
            v['seed_id']      = seed_id
            v['generated_at'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
            v['split']        = 'train'
            out_f.write(json.dumps(v, ensure_ascii=False) + '\n')
            written += 1

        with cp_lock:
            state['stats']['augmented'] += written
            if not variants: state['stats']['failed'] += 1
            done_set.add(seed_id)
            state['done_ids'].append(seed_id)

        if (idx + 1) % SAVE_EVERY == 0 or idx + 1 == len(pending):
            save_checkpoint(state, upload=(idx+1) % (SAVE_EVERY*5) == 0)
            print(f'  [{idx+1}/{len(pending)}] augmented={state["stats"]["augmented"]} deduped={state["stats"]["deduped"]}')

        time.sleep(0.5)

total_aug   = sum(1 for _ in open(AUGMENTED_PATH, encoding='utf-8')) if AUGMENTED_PATH.exists() else 0
total_seeds = len(seeds)
print(f'\n[p3b] seeds={total_seeds} augmented_variants={total_aug} total_corpus={total_seeds+total_aug}')
save_checkpoint(state, upload=True)
print('[done] ready for p3c_upload.ipynb')